# Bedrock Embeddings을 활용해서 임베딩 계산

In [1]:
from langchain.embeddings import BedrockEmbeddings
from numpy import dot
from numpy.linalg import norm

#Bedrock Embeddings LangChain 클라이언트 생성
#https://api.python.langchain.com/en/latest/embeddings/langchain_community.embeddings.bedrock.BedrockEmbeddings.html
emb = BedrockEmbeddings()

class EmbedItem:
    def __init__(self, text):
        self.text = text
        self.embedding = emb.embed_query(text)

class ComparisonResult:
    def __init__(self, text, similarity):
        self.text = text
        self.similarity = similarity

def calculate_similarity(a, b): # 코사인 유사도를 확인하세요: https://en.wikipedia.org/wiki/Cosine_similarity
    return dot(a, b) / (norm(a) * norm(b))


## 텍스트 embedding 

In [2]:
embeded_text = emb.embed_query("Hello")
print("vector size = ", len(embeded_text))
print(embeded_text)

vector size =  1536
[0.053955078125, -0.34375, -0.15625, 0.458984375, 1.3046875, 0.4140625, 0.490234375, -0.0009613037109375, -0.51171875, -0.271484375, 0.71875, 0.96875, 0.2216796875, -0.32421875, 0.34375, -0.2333984375, -0.1591796875, -0.0986328125, -1.0546875, 0.56640625, -0.12451171875, 0.34765625, 0.2138671875, -0.1337890625, -0.40234375, -0.20703125, 1.15625, -0.84375, 0.2109375, -0.6953125, 0.439453125, 1.0390625, 0.361328125, -0.6875, -0.26171875, 0.306640625, 0.63671875, -0.01544189453125, 0.81640625, 0.1171875, 0.474609375, -0.63671875, 1.0546875, 1.21875, 0.89453125, -0.037109375, -0.049560546875, -0.053466796875, 0.47265625, 1.375, 0.478515625, 0.7578125, -0.7109375, -0.11767578125, 0.08203125, -0.482421875, -0.1474609375, -0.126953125, 0.298828125, -0.2890625, -0.12451171875, -0.42578125, -0.10693359375, 0.1572265625, 0.353515625, -0.90234375, -1.0703125, 0.1376953125, -0.404296875, 0.296875, -0.25390625, 0.040771484375, 0.5546875, 1.171875, 0.408203125, -0.0927734375, 0.8

### 텍스트 파일 로드

In [14]:
#비교할 임베딩 목록을 작성합니다.
items = []

with open("items.txt", "r") as f:
    text_items = f.read().splitlines()

for text in text_items:
    items.append(EmbedItem(text))

# 디버그 메시지 출력
for item in items:
    print("Text:", item.text)
    print("Embedding:", item.embedding)
    print("-" * 20)

Text: Felines, canines, and rodents
Embedding: [0.26953125, -0.0703125, 0.375, 0.2041015625, 0.036376953125, 0.310546875, 0.0859375, -0.00092315673828125, 0.015380859375, -0.515625, 0.416015625, -0.1337890625, -0.14453125, -0.0224609375, 0.1015625, -0.09765625, 0.671875, 0.02734375, -0.10546875, 0.68359375, -0.267578125, -0.357421875, -0.12060546875, -0.023193359375, -0.0634765625, 0.79296875, 0.039306640625, -0.2333984375, 0.39453125, -0.10693359375, 0.82421875, -0.671875, 0.0634765625, -1.0859375, 0.4140625, -0.498046875, 0.33203125, 0.134765625, 0.75390625, 0.00311279296875, -0.318359375, 0.6953125, 1.109375, -0.294921875, -0.890625, 0.302734375, -0.37109375, 0.4609375, 0.287109375, -0.205078125, 0.333984375, 0.474609375, -0.22265625, -0.8359375, -0.7265625, -0.0186767578125, -0.2216796875, 0.1494140625, -0.57421875, -0.3671875, 0.267578125, 0.35546875, 0.62890625, -0.6171875, -0.09619140625, 0.30859375, 0.62890625, -0.12109375, -1.078125, 0.06591796875, -0.54296875, -0.74609375, 0.

## 텍스트 사이의 Cosine 유사도 계산

In [15]:
for e1 in items:
    print(f"Closest matches for '{e1.text}'")
    print ("----------------")
    cosine_comparisons = []
    
    for e2 in items:
        similarity_score = calculate_similarity(e1.embedding, e2.embedding) # 두 문장간 코사인 유사도를 구하고
        
        cosine_comparisons.append(ComparisonResult(e2.text, similarity_score)) # 코사인 유사도 값 을 목록에 저장합니다.
        
    cosine_comparisons.sort(key=lambda x: x.similarity, reverse=True) # 가장 가까운 일치 항목을 먼저 나열합니다.
    # 일반 함수 버전
    # def get_similiarity(x):
    #     return x.similarity
    # cosine_comparisons.sort(key=get_similarity, reverse=True)
    
    for c in cosine_comparisons:
        print("%.6f" % c.similarity, "\t", c.text)
    
    print()

Closest matches for 'Felines, canines, and rodents'
----------------
1.000000 	 Felines, canines, and rodents
0.872856 	 Cats, dogs, and mice
0.599730 	 Chats, chiens et souris
0.516598 	 Lions, tigers, and bears
0.456268 	 고양이, 개, 쥐
0.455923 	 猫、犬、ネズミ
0.068916 	 パン屋への道順を知りたい
0.061314 	 パン屋への行き方を教えてください
0.034925 	 빵집으로 가는 길을 알려주세요.
0.024160 	 경기장 가는 방법을 알려주시겠어요?
0.002239 	 Can you please tell me how to get to the stadium?
-0.003159 	 Kannst du mir bitte sagen, wie ich zur Bäckerei komme?
-0.007595 	 Can you please tell me how to get to the bakery?
-0.019469 	 Pouvez-vous s'il vous plaît me dire comment me rendre à la boulangerie?
-0.020840 	 I need directions to the bread shop

Closest matches for 'Can you please tell me how to get to the bakery?'
----------------
1.000000 	 Can you please tell me how to get to the bakery?
0.712236 	 I need directions to the bread shop
0.541959 	 Pouvez-vous s'il vous plaît me dire comment me rendre à la boulangerie?
0.492384 	 빵집으로 가는 길을 알려주세요.
0.4846

### (참조)Titan Embedding v2.0

In [16]:
import json
import boto3 
bedrock_runtime = boto3.client("bedrock-runtime")

vector_size = 256 ## 1024, 512, 256

body = json.dumps(
    {
        "inputText": "this is where you place your input text",
        "dimensions": 256,
        "normalize": True
    }
)

response = bedrock_runtime.invoke_model(
    body=body, 
    modelId="amazon.titan-embed-text-v2:0", 
    accept="application/json", 
    contentType="application/json"
)
response_body = json.loads(response.get("body").read())

print(len(response_body.get("embedding")))
print(response_body)

256
{'embedding': [-0.11338554322719574, 0.03379761427640915, -0.03794054687023163, -0.01875222474336624, 0.10161089152097702, -0.03314346820116043, 0.055384475737810135, 0.058873262256383896, -0.05276788771152496, 0.04491811990737915, -0.11164115369319916, 0.08198647201061249, 0.10946065932512283, 0.036850303411483765, -0.06367034465074539, -0.03205322101712227, 0.01962442137300968, 0.037722498178482056, 0.05494837835431099, -0.038376644253730774, -0.009430624544620514, 0.010248309001326561, 0.015590512193739414, 0.014064168557524681, -0.1883944422006607, -0.025620771571993828, -0.08547525852918625, -0.05930935963988304, 0.1360626518726349, -0.09201672673225403, 0.04317372664809227, 0.08155037462711334, 0.06977572292089462, -0.0064324489794671535, 0.0178800281137228, 0.09812210500240326, 0.09986649453639984, 0.02791028842329979, 0.0022077474277466536, 0.04339177533984184, 0.0627981498837471, 0.033361516892910004, -0.002166863065212965, 0.08721964806318283, 0.0715201124548912, -0.07588